# Bayesian Optimization Analysis — nibs_bo_001

Live analysis of the Optuna study stored in `nibs_bo_001.db`.
Re-run cells as trials complete to see updated results.

In [ ]:
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['figure.dpi'] = 100

STUDY_NAME = "nibs_bo_001"
DB_PATH = "sqlite:///nibs_bo_001.db"

study = optuna.load_study(study_name=STUDY_NAME, storage=DB_PATH)
df = study.trials_dataframe()
df = df[df["state"] == "COMPLETE"].copy()
print(f"Completed trials: {len(df)}")
print(f"Best value: {study.best_value:.2f} €  (trial #{study.best_trial.number})")

## Convergence Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: all trial values + running best
ax = axes[0]
vals = df["value"].values
best_so_far = np.minimum.accumulate(vals)
ax.scatter(range(len(vals)), vals, s=12, alpha=0.5, label="Trial cost")
ax.plot(best_so_far, color="red", linewidth=2, label="Best so far")
ax.set_xlabel("Trial #")
ax.set_ylabel("Total Cost (€)")
ax.set_title("Convergence")
ax.legend()

# Right: cost breakdown of best-so-far trials
ax = axes[1]
energy = df["user_attrs_energy_cost_eur"].values
co2 = df["user_attrs_co2_penalty_eur"].values
temp = df["user_attrs_temp_penalty_eur"].values

ax.bar(range(len(vals)), energy, label="Energy", alpha=0.8)
ax.bar(range(len(vals)), co2, bottom=energy, label="CO2 penalty", alpha=0.8)
ax.bar(range(len(vals)), temp, bottom=energy + co2, label="Temp penalty", alpha=0.8)
ax.set_xlabel("Trial #")
ax.set_ylabel("Cost (€)")
ax.set_title("Cost Breakdown per Trial")
ax.legend()

plt.tight_layout()
plt.show()

## Best Trials — Top 10

In [ ]:
top = df.nsmallest(10, "value")[
    ["number", "value", "user_attrs_energy_cost_eur",
     "user_attrs_co2_penalty_eur", "user_attrs_temp_penalty_eur"]
].copy()
top.columns = ["Trial", "Total €", "Energy €", "CO2 €", "Temp €"]
top = top.set_index("Trial")
top.style.format("{:.1f}").background_gradient(cmap="RdYlGn_r", subset=["Total €"])

## Parameter Importance

Which parameters have the biggest effect on the objective?

In [ ]:
try:
    importance = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame({"param": importance.keys(), "importance": importance.values()})
    imp_df = imp_df.sort_values("importance", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(imp_df["param"], imp_df["importance"])
    ax.set_xlabel("Importance")
    ax.set_title("Parameter Importance (fANOVA)")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Need more trials for importance analysis: {e}")

## Parameter Distributions — Top 20 vs All Trials

Compare parameter values in the best 20% of trials vs the full population.

In [ ]:
param_cols = [c for c in df.columns if c.startswith("params_")]
n_top = max(2, len(df) // 5)  # top 20%
top_df = df.nsmallest(n_top, "value")

n_params = len(param_cols)
n_cols = 4
n_rows = (n_params + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(sorted(param_cols)):
    ax = axes[i]
    name = col.replace("params_", "")
    ax.hist(df[col], bins=20, alpha=0.4, label="All", density=True)
    ax.hist(top_df[col], bins=15, alpha=0.6, label=f"Top {n_top}", density=True)
    ax.set_title(name, fontsize=9)
    ax.legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Parameter Distributions: All vs Top 20%", fontsize=13)
plt.tight_layout()
plt.show()

## Cost Component Correlations

How do the three cost components relate to each other?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cost_cols = {
    "Energy": "user_attrs_energy_cost_eur",
    "CO2": "user_attrs_co2_penalty_eur",
    "Temp": "user_attrs_temp_penalty_eur",
}

pairs = [("Energy", "Temp"), ("Energy", "CO2"), ("CO2", "Temp")]
for ax, (x_name, y_name) in zip(axes, pairs):
    ax.scatter(df[cost_cols[x_name]], df[cost_cols[y_name]],
               c=df["value"], cmap="viridis_r", s=20, alpha=0.7)
    ax.set_xlabel(f"{x_name} (€)")
    ax.set_ylabel(f"{y_name} (€)")
    ax.set_title(f"{x_name} vs {y_name}")

plt.tight_layout()
plt.show()

## Best Trial Parameters

In [ ]:
best = study.best_trial
print(f"Best trial #{best.number} — Total: {best.value:.2f} €")
print(f"  Energy:  {best.user_attrs.get('energy_cost_eur', '?'):.2f} €")
print(f"  CO2:     {best.user_attrs.get('co2_penalty_eur', '?'):.2f} €")
print(f"  Temp:    {best.user_attrs.get('temp_penalty_eur', '?'):.2f} €")
print()
for k, v in sorted(best.params.items()):
    print(f"  {k:30s} = {v:.4f}")

## Suggested Search Space Narrowing for nibs_bo_002

Based on top-performing trials, suggest tighter ranges.

In [ ]:
n_top = max(3, len(df) // 5)
top_df = df.nsmallest(n_top, "value")

print(f"Ranges from top {n_top} trials (suggested tighter bounds):\n")
for col in sorted(param_cols):
    name = col.replace("params_", "")
    lo, hi = top_df[col].min(), top_df[col].max()
    margin = (hi - lo) * 0.2  # 20% margin
    print(f"  {name:30s}  [{lo - margin:.2f}, {hi + margin:.2f}]  (was [{df[col].min():.2f}, {df[col].max():.2f}])")